# 실전 13-2: 문제 분해 및 계획 (Least-to-Most & Plan-and-Solve)

## 1. Least-to-Most Prompting 이란?
- "복잡한 문제는 쪼개서 풀어라" 라는 철학입니다.
- LLM에게 벅찬 거대한 질문을 던지면 환각이 발생합니다. 먼저 하위 문제(Sub-problems)들로 분해하게 한 다음, 앞선 하위 문제의 답을 활용하여 다음 하위 문제를 순차적으로 푸는 기법입니다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Least-to-Most 체인 구성하기
예를 들어 "영국에서 가장 긴 강을 지나는 도시 중 인구가 가장 많은 도시의 시장 이름은?" 이라는 복잡한 질문이 있습니다.

In [2]:
complex_question = "프랑스 혁명이 일어난 연도에 태어난 조선의 왕의 이름과 그의 재위 기간은?"

# 1. 분해 노드: 복잡한 질문을 리스트로 쪼갭니다.
decomp_prompt = ChatPromptTemplate.from_template("다음 질문을 풀기 위해 순서대로 알아내야 할 하위 질문 3가지를 리스트 형태로 적어줘. 답은 적지 마: {question}")
decomp_chain = decomp_prompt | llm

sub_questions = decomp_chain.invoke({"question": complex_question})
print("=== [1단계: 하위 문제로 분해] ===")
print(sub_questions.content)

# 2. 순차 해결 노드 (실제로는 LangGraph의 Loop로 각 하위 질문을 하나씩 검색해서 풀게 만들지만, 여기서는 컨셉만 보여줍니다.)
solve_prompt = ChatPromptTemplate.from_template("당신이 세운 다음 하위 질문 리스트를 보고, 1번부터 차근차근 답을 구하여 최종 결론(조선의 왕과 재위 기간)을 도출해: \n\n[하위 질문들]:\n{sub_q}")
solve_chain = solve_prompt | llm

print("\n=== [2단계: 쪼갠 문제를 순차적으로 해결] ===")
final_answer = solve_chain.invoke({"sub_q": sub_questions.content})
print(final_answer.content)

=== [1단계: 하위 문제로 분해] ===
1. 프랑스 혁명이 일어난 연도는 언제인가?
2. 그 연도에 태어난 조선의 왕은 누구인가?
3. 해당 왕의 재위 기간은 언제부터 언제까지인가?

=== [2단계: 쪼갠 문제를 순차적으로 해결] ===
하위 질문에 따라 차근차근 답을 구해보겠습니다.

1. **프랑스 혁명이 일어난 연도는 언제인가?**
   - 프랑스 혁명은 1789년에 시작되었습니다.

2. **그 연도에 태어난 조선의 왕은 누구인가?**
   - 1789년에 태어난 조선의 왕은 고종(이재곤)입니다. 그는 1852년에 태어났지만, 1789년은 고종의 아버지인 철종이 재위하던 시기입니다. 고종은 1863년에 왕위에 올랐습니다.

3. **해당 왕의 재위 기간은 언제부터 언제까지인가?**
   - 고종의 재위 기간은 1863년부터 1907년까지입니다.

최종 결론:
- 조선의 왕 고종은 1863년부터 1907년까지 재위하였습니다.


## 3. Plan-and-Solve (계획 세우고 해결하기)
- 코딩 에이전트나 보고서 작성 AI에게 가장 많이 쓰이는 기법입니다.
- "파이썬으로 테트리스 게임 만들어줘" 라고 하면 쓰레기 코드를 내뱉지만,
- "1. 설계도(목차/계획)를 세워라 -> 2. 그 계획에 따라 1번부터 코드를 작성해라" 라고 시키면 매우 훌륭한 결과물이 나옵니다.

In [3]:
task = "파이썬으로 로또 번호 6개를 뽑고, 1등 당첨 번호(임의지정)와 비교해서 몇 등인지 출력하는 프로그램 짜기"

print("=== [일반 방식의 실패 확률] ===")
print("그냥 짜라고 하면 보통 한 번에 훅 짜버리며 로직 에러가 나기 쉽습니다.\n")

print("=== [Plan-and-Solve 기법 적용] ===")
plan_solve_prompt = ChatPromptTemplate.from_template(
    "당신은 수석 아키텍트입니다.\n"
    "주어진 요구사항을 구현하기 위해:\n"
    "1. [계획]: 문제를 해결하기 위한 구체적인 단계별 계획(목차)을 먼저 세우세요.\n"
    "2. [해결]: 세운 계획에 따라 차례대로 로직을 구현하여 최종 파이썬 코드를 작성하세요.\n\n"
    "요구사항: {task}"
)
plan_solve_chain = plan_solve_prompt | llm

result = plan_solve_chain.invoke({"task": task})
print(result.content)

=== [일반 방식의 실패 확률] ===
그냥 짜라고 하면 보통 한 번에 훅 짜버리며 로직 에러가 나기 쉽습니다.

=== [Plan-and-Solve 기법 적용] ===
## [계획]

1. **필요한 라이브러리 임포트**
   - 랜덤 번호 생성을 위해 `random` 라이브러리 임포트

2. **로또 번호 생성 함수 정의**
   - 1부터 45까지의 숫자 중에서 6개의 고유한 번호를 랜덤으로 선택하는 함수 정의

3. **1등 당첨 번호 설정**
   - 임의로 1등 당첨 번호를 설정

4. **로또 번호와 당첨 번호 비교 함수 정의**
   - 생성된 로또 번호와 1등 당첨 번호를 비교하여 몇 등인지 판단하는 함수 정의

5. **메인 함수 정의**
   - 로또 번호를 생성하고, 당첨 번호와 비교하여 결과 출력

6. **프로그램 실행**
   - 메인 함수를 호출하여 프로그램 실행

## [해결]

아래는 위의 계획에 따라 작성한 파이썬 코드입니다.

```python
import random

def generate_lotto_numbers():
    """1부터 45까지의 숫자 중에서 6개의 고유한 번호를 랜덤으로 선택"""
    return random.sample(range(1, 46), 6)

def compare_with_winning_numbers(lotto_numbers, winning_numbers):
    """로또 번호와 당첨 번호를 비교하여 몇 등인지 판단"""
    matched_numbers = set(lotto_numbers) & set(winning_numbers)
    match_count = len(matched_numbers)
    
    if match_count == 6:
        return "1등"
    elif match_count == 5:
        return "2등"
    elif match_count == 4:
        return "3등"
    elif match

## 4. 결론
- 복잡한 쿼리나 태스크를 만났을 때 **"작은 문제로 쪼개기(Decomposition)"** 와 **"계획 먼저 세우기(Planning)"** 는 에이전트를 영리하게 만드는 치트키입니다.